# UNIDAD IV — Manejo de archivos, persistencia y validación

## CLASE 4: Proyecto Final – Sistema de Gestión con Persistencia en Archivos

---

# PARTE I — FUNDAMENTOS CONCEPTUALES DEL PROYECTO


---



# 1. Qué es un sistema de información

Un programa simple:

```
Hace algo → termina
```

Un sistema:

```
Recibe datos → los valida → los guarda → los modifica → los consulta → mantiene historial
```

El objetivo de esta clase:

> Construir un sistema, no un script

---


# 2. Modelo del problema: gestión de contactos

Un contacto tiene estructura:

| Campo    | Tipo   | Restricción    |
| -------- | ------ | -------------- |
| id       | entero | único          |
| nombre   | texto  | obligatorio    |
| telefono | texto  | formato válido |
| email    | texto  | válido         |
| activo   | bool   | borrado lógico |

---

# 3. Operaciones del sistema

El sistema debe permitir:

* Añadir
* Buscar
* Editar
* Eliminar (soft delete)
* Listar
* Guardar en CSV
* Restaurar desde archivo
* Exportar backup

---


# 4. Flujo profesional de datos

```
Usuario → Validación → Servicio → Repositorio → CSV → Disco
                                     ↑
                              Manejo de errores
```

---

# 5. Arquitectura obligatoria

```
contact_app/
│
├── main.py
├── models/
│   └── contact.py
├── repository/
│   └── contact_repository.py
├── services/
│   └── contact_service.py
├── utils/
│   ├── csv_storage.py
│   ├── validators.py
│   └── backup.py
└── core/
    └── errors.py
```

---


---

# PARTE II — IMPLEMENTACIÓN GUIADA


---

# 1. Modelo de dominio

## models/contact.py



In [ ]:
from dataclasses import dataclass

@dataclass
class Contact:
    id: int
    name: str
    phone: str
    email: str
    active: bool = True


---

# 2. Excepciones del sistema

## core/errors.py



In [ ]:
class ContactError(Exception):
    pass

class ContactNotFound(ContactError):
    pass

class DuplicateContact(ContactError):
    pass

class ValidationError(ContactError):
    pass

class StorageError(ContactError):
    pass


---

# 3. Validaciones

## utils/validators.py



In [ ]:
import re
from core.errors import ValidationError

def validate_email(email: str) -> str:
    if not re.match(r"[^@]+@[^@]+\.[^@]+", email):
        raise ValidationError("Email inválido")
    return email.strip()

def validate_phone(phone: str) -> str:
    if not re.match(r"^[0-9+\- ]{7,15}$", phone):
        raise ValidationError("Teléfono inválido")
    return phone.strip()

def validate_name(name: str) -> str:
    if not name.strip():
        raise ValidationError("Nombre vacío")
    return name.strip()


---

# 4. Infraestructura CSV segura

## utils/csv_storage.py



In [ ]:
import csv
from pathlib import Path
from core.errors import StorageError

HEADERS = ["id", "name", "phone", "email", "active"]

class CSVStorage:

    def __init__(self, path: str):
        self.path = Path(path)
        self.path.touch(exist_ok=True)

    def load(self):

        try:
            with self.path.open("r", encoding="utf-8", newline="") as f:
                reader = csv.DictReader(f)
                return list(reader)

        except Exception as e:
            raise StorageError("Error leyendo archivo") from e

    def save(self, rows):

        try:
            with self.path.open("w", encoding="utf-8", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=HEADERS)
                writer.writeheader()
                writer.writerows(rows)

        except Exception as e:
            raise StorageError("Error guardando archivo") from e


---

# 5. Repositorio

## repository/contact_repository.py



In [ ]:
from models.contact import Contact
from utils.csv_storage import CSVStorage
from core.errors import ContactNotFound

class ContactRepository:

    def __init__(self, storage: CSVStorage):
        self.storage = storage

    def _map(self, row):
        return Contact(
            int(row["id"]),
            row["name"],
            row["phone"],
            row["email"],
            row["active"] == "True"
        )

    def get_all(self):
        return [self._map(r) for r in self.storage.load()]

    def save_all(self, contacts):
        rows = [c.__dict__ for c in contacts]
        self.storage.save(rows)

    def find(self, id_):
        for c in self.get_all():
            if c.id == id_:
                return c
        raise ContactNotFound("Contacto no existe")


---

# 6. Servicio

## services/contact_service.py



In [ ]:
from core.errors import DuplicateContact, ContactNotFound
from utils.validators import validate_email, validate_phone, validate_name
from models.contact import Contact

class ContactService:

    def __init__(self, repo):
        self.repo = repo

    def add(self, id_, name, phone, email):

        contacts = self.repo.get_all()

        if any(c.id == id_ for c in contacts):
            raise DuplicateContact("ID ya existe")

        contact = Contact(
            id_,
            validate_name(name),
            validate_phone(phone),
            validate_email(email),
            True
        )

        contacts.append(contact)
        self.repo.save_all(contacts)

    def delete(self, id_):
        contacts = self.repo.get_all()
        for c in contacts:
            if c.id == id_:
                c.active = False
                self.repo.save_all(contacts)
                return
        raise ContactNotFound()

    def list_active(self):
        return [c for c in self.repo.get_all() if c.active]


---

# 7. Backup

## utils/backup.py



In [ ]:
from shutil import copyfile
from datetime import datetime

def create_backup(path: str):

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_name = f"{path}_{timestamp}.bak"
    copyfile(path, backup_name)


---

# 8. Interfaz consola

## main.py



In [ ]:
from utils.csv_storage import CSVStorage
from repository.contact_repository import ContactRepository
from services.contact_service import ContactService
from utils.backup import create_backup
from core.errors import ContactError

def main():

    service = ContactService(ContactRepository(CSVStorage("data/contacts.csv")))

    while True:

        print("\n1.Agregar 2.Listar 3.Eliminar 4.Backup 5.Salir")
        op = input("Opción: ")

        try:
            if op == "1":
                service.add(
                    int(input("ID: ")),
                    input("Nombre: "),
                    input("Teléfono: "),
                    input("Email: ")
                )

            elif op == "2":
                for c in service.list_active():
                    print(c)

            elif op == "3":
                service.delete(int(input("ID: ")))

            elif op == "4":
                create_backup("data/contacts.csv")

            elif op == "5":
                break

        except ContactError as e:
            print("Error:", e)

if __name__ == "__main__":
    main()


# PARTE III — TALLER PRÁCTICO

El sistema debe resistir:

1. ID duplicado
2. Email inválido
3. Teléfono incorrecto
4. Eliminación de inexistente
5. Archivo inexistente
6. Archivo corrupto
7. Backup funcionando

---
